In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
#guid 2888818L

In [4]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import numpy
from collections import Counter


In [10]:
def entangledPair():
    qc = QuantumCircuit(2,2)
    qc.h(0)
    qc.cx(0,1)
    qc.x(1)
    qc.z(1)
    return qc



def random():
    qc_rand = QuantumCircuit(1)
    qc_rand.append(Operator(uni_third_op), [0])
    qc_rand.measure_all()
    backend = BasicSimulator()
    qc_compiled = transpile(qc_rand, backend)
    job_sim = backend.run(qc_compiled, shots=1)
    result_sim = job_sim.result()
    unique = set(result_sim.get_counts(qc_compiled))
    result = int(unique.pop())
    if result == 0:
        return 1
    else:
        qc_ran = QuantumCircuit(1)
        qc_ran.h(0)
        qc_ran.measure_all()
        qc_compiled = transpile(qc_ran, backend)
        job_sim = backend.run(qc_compiled, shots=1)
        result_sim = job_sim.result()
        unique = set(result_sim.get_counts(qc_compiled))
        result = int(unique.pop())
        if result == 0:
            return 2
        else:
            return 3

root2 = math.sqrt(2)

denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2)

W_op = [ [ -1 / denom1 , (1 + root2) / denom1 ],
         [  1 / denom2 , (root2 - 1) / denom2 ] ]

V_op = [ [  1 / denom1 , (1 + root2) / denom1 ],
         [ -1 / denom2 , (root2 - 1) / denom2 ] ]


uni_third_op = [[1/math.sqrt(3), -math.sqrt(2/3)],
                [math.sqrt(2/3), 1/math.sqrt(3)]]

Alice_operation = {2:W_op}
Bob_operation = {1:W_op , 3:V_op}

Alice_key = []
Alice_base = []
Bob_key = []
Bob_base = []
final_key = []
bell_state = []


backend = BasicSimulator()

def create_key(N):
    global Alice_key, Bob_key, Alice_base, Bob_base, final_key, bell_state
    Alice_key = []
    Bob_key = []
    final_key = []
    Alice_base = []
    Bob_base = []
    bell_state = []

    for i in range((9*N)//2):
        pair = entangledPair()

        Alice_rand = random()
        Bob_rand = random()
        if (Alice_rand == 1):
          pair.h(0)
        elif (Alice_rand == 3):
          pass
        else:
          pair.append(Operator(Alice_operation[Alice_rand]), [0])

        if (Bob_rand == 2):
          pass
        else:
          pair.append(Operator(Bob_operation[Bob_rand]), [1])

        pair.measure_all()
        qc_compiled = transpile(pair, backend)
        job_sim = backend.run(qc_compiled, shots=1)
        result_sim = job_sim.result()

        unique = set(result_sim.get_counts(qc_compiled))
        unique = unique.pop()

        Alice_key.append(int(unique[1]))
        Bob_key.append(1 - int(unique[0]))
        Alice_base.append(Alice_rand)
        Bob_base.append(Bob_rand)

    for i in range(len(Alice_key)):
        if Alice_base[i] == 2 and Bob_base[i]==1 or Alice_base[i] ==3 and Bob_base[i]== 2:
            final_key.append(Alice_key[i])
        else:
            bell_state.append((Alice_key[i], Bob_key[i], Alice_base[i], Bob_base[i]))


def average():
  AX_BW_average = 0
  AX_BV_average = 0
  AZ_BW_average = 0
  AZ_BV_average = 0

  count = {"XW":0, "XV":0, "ZW":0, "ZV":0}

  for state in bell_state:
    if state[0] == 0:
      a_bit = 1
    else:
      a_bit = -1

    if state[1] == 0:
      b_bit = 1
    else:
      b_bit = -1

    if (state[2] == 1):
      if (state[3]==1):
        AX_BW_average += (a_bit * b_bit)
        count["XW"] += 1
      elif (state[3]==3):
        AX_BV_average += (a_bit * b_bit)
        count["XV"] += 1
    elif (state[2] == 3):
      if (state[3]==1):
        AZ_BW_average += (a_bit * b_bit)
        count["ZW"] += 1
      elif (state[3]==3):
        AZ_BV_average += (a_bit * b_bit)
        count["ZV"] += 1

  S = abs(AX_BW_average/count["XW"] - AX_BV_average/count["XV"]  + AZ_BW_average/count["ZW"]  + AZ_BV_average/count["ZV"] )
  print("|S| = " + str(S))

  if (S <= 2*root2 and S > 2):
    print("S < 2√2 but close enough to ensure quantum validity")

  elif (S <= 2):
    print("sever attack occured, loss of quantum correlation")

N = 40
create_key(N)
print("Alice key: " + str(Alice_key))
print("Bob key: " + str(Bob_key))
print("Shared key: " + str(final_key))
average()

N = 40
create_key(N)
print("Alice key: " + str(Alice_key))
print("Bob key: " + str(Bob_key))
print("Shared key: " + str(final_key))
average()

N = 40
create_key(N)
print("Alice key: " + str(Alice_key))
print("Bob key: " + str(Bob_key))
print("Shared key: " + str(final_key))
average()

Alice key: [1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1]
Bob key: [1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 

In [8]:

q = QuantumCircuit(2)
q.h(0)
q.cx(0,1)
q.x(1)
q.z(1)
state = Statevector.from_int(0, 4)
state = state.evolve(q)
state.draw("latex")


<IPython.core.display.Latex object>

In [13]:
c1 = 0
c2 = 0
c3 = 0

for i in range(21):
    r = random()
    if r == 1:
        c1 +=1
    elif r == 2:
        c2+=1
    else:
        c3+=1
print(str(c1)+ ":"+str(c2)+":"+str(c3))

7:7:7
